# 03 · Layer 1：Model 設定與結構化輸出

前兩章我們一直用 `get_model()` 帶過模型設定。這一章把它拆開：

1. 模型「字串」與模型「物件」差在哪，什麼時候非用物件不可
2. 用 `GenerateContentConfig` 控制生成行為（temperature、長度、安全性）
3. 接非 Gemini 的模型（LiteLLM）
4. **`output_schema`：讓 agent 吐出保證合法的 JSON** ← 串接系統時最重要的一節

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. 三種指定模型的方式

| 寫法 | 適用 | 缺點 |
|---|---|---|
| `model="gemini-2.5-flash"` | 快速試作 | 沒地方掛重試、base_url 等設定 |
| `model=Gemini(model=..., retry_options=...)` | 正式使用 | 多兩行 |
| `model=LiteLlm(model="...")` | 非 Gemini 模型 | 需要額外套件與端點 |

In [2]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini

# 寫法 A：字串
agent_a = LlmAgent(name="a", model="gemini-2.5-flash", instruction="簡短回答。")

# 寫法 B：物件（本教材 get_model() 用的就是這個）
agent_b = LlmAgent(
    name="b",
    model=Gemini(model="gemini-2.5-flash"),
    instruction="簡短回答。",
)

print("A:", type(agent_a.model).__name__, "|", agent_a.model)
print("B:", type(agent_b.model).__name__, "|", agent_b.model.model)

A: str | gemini-2.5-flash
B: Gemini | gemini-2.5-flash


兩種寫法 ADK 都吃。差別在**字串沒有地方掛設定**。

最實際的例子是重試：AI Studio 免費層很容易撞到 429（配額）和 503（模型忙碌），
而 **ADK 預設不會重試**。一本 notebook 跑到一半斷掉，通常就是這個原因。

In [3]:
from google.genai.types import HttpRetryOptions

resilient = Gemini(
    model="gemini-2.5-flash",
    retry_options=HttpRetryOptions(
        attempts=5,
        initial_delay=2.0,
        max_delay=30.0,
        http_status_codes=[429, 500, 502, 503, 504],
    ),
)
print("重試設定 :", resilient.retry_options)

重試設定 : attempts=5 initial_delay=2.0 max_delay=30.0 exp_base=None jitter=None http_status_codes=[429, 500, 502, 503, 504]


> `shared/config.py` 的 `get_model()` 已經幫你掛好這組設定了，
> 所以本教材的每個 agent 都是有重試保護的。

### 模型會下架

這不是理論問題。`gemini-2.0-flash` 已經停用，呼叫它會直接得到 404：

In [4]:
try:
    dead = LlmAgent(name="dead", model="gemini-2.0-flash", instruction="hi")
    await run_once(dead, "hello")
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:260]}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'stat


所以模型 ID 要集中管理。本教材全部集中在 `shared/config.py` 的
`DEFAULT_MODEL`，真的哪天壞了只要改一行。

## 2. `generate_content_config`：控制生成行為

最常調的是 `temperature`：

- `0.0` → 幾乎每次都給一樣的答案。**抽取資料、分類、走流程時用這個。**
- `1.0+` → 發散、有創意。寫文案、腦力激盪時用。

In [5]:
from google.genai import types

def make_writer(temperature: float) -> LlmAgent:
    return LlmAgent(
        name=f"writer_t{str(temperature).replace('.', '')}",
        model=get_model(),
        instruction="你是文案寫手。為使用者給的產品寫一句廣告標語，只回標語本身。",
        generate_content_config=types.GenerateContentConfig(temperature=temperature),
    )


prompt = "一款專門給工程師的保溫杯"

print("temperature = 0.0（穩定）")
for i in range(3):
    print(f"  {i + 1}. {await run_once(make_writer(0.0), prompt)}")

print("\ntemperature = 1.5（發散）")
for i in range(3):
    print(f"  {i + 1}. {await run_once(make_writer(1.5), prompt)}")

temperature = 0.0（穩定）


  1. 冷的是程式，熱的是堅持。


  2. 程式碼會過期，但這杯熱度永遠線上。


  3. 冷氣房裡的熱血，只有它懂。

temperature = 1.5（發散）


  1. 程式碼會冷，但你的咖啡永遠滾燙。


  2. 程式碼會涼，你的熱情不會。


  3. 冷的是程式，熱的是堅持，裝滿靈感，隨時滿血。


`GenerateContentConfig` 還有幾個常用欄位：

| 欄位 | 用途 |
|---|---|
| `max_output_tokens` | 硬性截斷輸出長度（省錢） |
| `top_p` / `top_k` | 另外兩個控制隨機性的旋鈕 |
| `safety_settings` | 安全過濾強度，見第 28 天 |
| `thinking_config` | 控制 thinking 模型要想多久 |

In [6]:
short = LlmAgent(
    name="short_bot",
    model=get_model(),
    instruction="回答使用者的問題。",
    generate_content_config=types.GenerateContentConfig(max_output_tokens=40),
)
print(await run_once(short, "請詳細說明什麼是遞迴，越詳細越好"))

你好！我是 **short_bot**。關於「遞迴（Recursion）」這個電腦科學與數學中的核心概念，我將為你進行極詳細的深度解析


注意輸出是**被硬切斷**的，不是模型「自己講得比較短」。
`max_output_tokens` 是預算上限，不是風格指示。

## 3. 接非 Gemini 的模型

ADK 透過 **LiteLLM** 支援上百種模型：Claude、GPT、本地的 Ollama / vLLM 都可以。

下面這個 cell **不會執行**（需要你自己的端點），列出來當參考：

In [7]:
# ↓↓↓ 說明用，不會執行 ↓↓↓
example_code = '''
from google.adk.models.lite_llm import LiteLlm

# Claude（需要 ANTHROPIC_API_KEY）
claude_agent = LlmAgent(
    name="claude_bot",
    model=LiteLlm(model="anthropic/claude-sonnet-4-5"),
    instruction="...",
)

# 本地 Ollama：注意前綴要用 ollama_chat 而不是 ollama，
# 用 ollama 會失去 function calling 能力
local_agent = LlmAgent(
    name="local_bot",
    model=LiteLlm(model="ollama_chat/llama3.1"),
    instruction="...",
)

# 本地 vLLM / OpenAI-compatible 端點
vllm_agent = LlmAgent(
    name="vllm_bot",
    model=LiteLlm(
        model="openai/openai/gpt-oss-120b",  # 第一個 openai/ 是 LiteLLM 路由前綴，會被剝掉
        api_base="http://localhost:5052/v1",  # 必須含 /v1
        api_key="dummy",
    ),
    instruction="...",
)
'''
print(example_code)


from google.adk.models.lite_llm import LiteLlm

# Claude（需要 ANTHROPIC_API_KEY）
claude_agent = LlmAgent(
    name="claude_bot",
    model=LiteLlm(model="anthropic/claude-sonnet-4-5"),
    instruction="...",
)

# 本地 Ollama：注意前綴要用 ollama_chat 而不是 ollama，
# 用 ollama 會失去 function calling 能力
local_agent = LlmAgent(
    name="local_bot",
    model=LiteLlm(model="ollama_chat/llama3.1"),
    instruction="...",
)

# 本地 vLLM / OpenAI-compatible 端點
vllm_agent = LlmAgent(
    name="vllm_bot",
    model=LiteLlm(
        model="openai/openai/gpt-oss-120b",  # 第一個 openai/ 是 LiteLLM 路由前綴，會被剝掉
        api_base="http://localhost:5052/v1",  # 必須含 /v1
        api_key="dummy",
    ),
    instruction="...",
)



### 三個實際會踩到的坑

1. **Ollama 要用 `ollama_chat/` 前綴**，不是 `ollama/`。用錯的話工具呼叫會失效。
2. **vLLM 要加 `--enable-auto-tool-choice`** 啟動參數，否則模型不會發出 function call。
3. **模型字串的雙前綴**：`openai/openai/gpt-oss-120b` 看起來像打錯，其實第一個
   `openai/` 是 LiteLLM 的協定路由（會被剝掉），剩下的才是端點上真正的模型 ID。

> 本專案的 `.env` 保留了 `ADK_PROVIDER=litellm` 這條路。把它打開，
> 前面每一章的程式碼完全不用改就會走本地模型——這就是把模型設定
> 集中在 `shared/config.py` 的好處。

## 4. `output_schema`：保證合法的結構化輸出

前面所有 agent 回的都是自然語言。但如果 agent 的下游是**程式**而不是人，
你需要的是 JSON。

「請你回 JSON」這種寫在 instruction 裡的做法不可靠——模型會加上
```` ```json ```` 圍籬、會多寫一句「以下是結果：」、偶爾會漏欄位。

`output_schema` 用 Pydantic 模型把格式**強制**下來。

In [8]:
from pydantic import BaseModel, Field


class Sentiment(BaseModel):
    """一則評論的分析結果。"""

    sentiment: str = Field(description="情緒，只能是 positive / neutral / negative 其中之一")
    score: float = Field(description="信心分數，0 到 1 之間")
    keywords: list[str] = Field(description="影響判斷的關鍵詞，最多三個")
    summary: str = Field(description="一句話摘要，繁體中文")


analyzer = LlmAgent(
    name="sentiment_analyzer",
    model=get_model(),
    instruction="你是評論分析器。分析使用者給的評論。",
    output_schema=Sentiment,
    output_key="analysis",
)

review = "這家店東西還不錯吃，但等了快一個小時，服務生態度也很差，不會再來了。"
raw = await run_once(analyzer, review)
print("原始輸出:")
print(raw)

原始輸出:
{
  "sentiment": "negative",
  "score": 0.95,
  "keywords": [
    "服務生態度差",
    "等了一個小時",
    "不會再來了"
  ],
  "summary": "食物雖可但等候時間過長且服務態度差，消費者表示不會再光顧。"
}


輸出是純 JSON，沒有圍籬、沒有客套話。直接 parse 就能用：

In [9]:
import json

data = Sentiment.model_validate_json(raw)
print(f"情緒     : {data.sentiment}")
print(f"信心     : {data.score}")
print(f"關鍵詞   : {', '.join(data.keywords)}")
print(f"摘要     : {data.summary}")

情緒     : negative
信心     : 0.95
關鍵詞   : 服務生態度差, 等了一個小時, 不會再來了
摘要     : 食物雖可但等候時間過長且服務態度差，消費者表示不會再光顧。


### `output_key` 把結果寫進 state

加上 `output_key="analysis"` 之後，結果會自動存進 session state。
這是多 agent pipeline 傳資料的標準做法（第 07、08 章的主軸）。

In [10]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=analyzer, app_name="concept_track")
sid = await new_session(runner)
await ask(runner, review, session_id=sid)

print_state(await peek_state(runner, sid))

  analysis: {'sentiment': 'negative', 'score': 0.95, 'keywords': ['態度差', '等太久', '不會再來'], 'summary': '餐點雖可但等候過久且服務態度差，消費者表示不會再訪。'}


## 5. `output_schema` 可以跟 `tools` 並存嗎？

這在舊版 ADK 是**不行**的——同時給 `output_schema` 和 `tools` 會直接拋錯，
網路上很多文章還是這樣寫。ADK 2.x 放寬了這個限制，我們實測一次：

In [11]:
def lookup_order(order_id: str) -> dict:
    """查詢訂單狀態。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "status": "已出貨", "eta": "2026-09-05"}


class OrderReply(BaseModel):
    order_id: str
    status: str = Field(description="訂單狀態")
    message: str = Field(description="給客戶看的一句話，繁體中文")


try:
    combo = LlmAgent(
        name="combo_agent",
        model=get_model(),
        instruction="你是客服。先用 lookup_order 查詢，再依 schema 回覆。",
        tools=[lookup_order],
        output_schema=OrderReply,
    )
    print("建立成功，實際跑跑看：")
    print(await run_once(combo, "幫我查訂單 A-1234", trace=True))
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:300]}")

建立成功，實際跑跑看：


  🔧 [combo_agent] 呼叫 lookup_order({'order_id': 'A-1234'})
  ↩️  [combo_agent] lookup_order 回傳 {'order_id': 'A-1234', 'status': '已出貨', 'eta': '2026-09-05'}


  🔧 [combo_agent] 呼叫 set_model_response({'status': '已出貨', 'order_id': 'A-1234', 'message': '您的訂單 A-1234 已經出貨，預計送達日期為 2026-09-05。'})
  ↩️  [combo_agent] set_model_response 回傳 {'order_id': 'A-1234', 'status': '已出貨', 'message': '您的訂單 A-1234 已經出貨，預計送達日期為 2026-09-05。'}
  💬 [combo_agent] {"order_id": "A-1234", "status": "已出貨", "message": "您的訂單 A-1234 已經出貨，預計送達日期為 2026-09-05。"}
{"order_id": "A-1234", "status": "已出貨", "message": "您的訂單 A-1234 已經出貨，預計送達日期為 2026-09-05。"}


在 ADK 2.8 上這是**允許**的。不過要注意兩件事：

1. 這是相對新的行為，跨版本／跨語言（Go、TS）不保證一致。
2. 更保守也更常見的做法是**拆成兩個 agent**：一個負責查、一個負責格式化，
   用 `SequentialAgent` 串起來（第 07 章）。

## 本章重點

- **字串 vs 物件**：字串短，但沒地方掛重試。ADK **預設不重試**，
  免費層很容易 429 / 503，正式使用一定要給 `retry_options`。
- **模型會下架**（`gemini-2.0-flash` 已停用），模型 ID 要集中管理。
- **`temperature`**：抽資料、走流程用 0；寫文案用 1 以上。
- **`max_output_tokens` 是硬切，不是風格指示。**
- **LiteLLM 三個坑**：`ollama_chat/` 前綴、vLLM 的 `--enable-auto-tool-choice`、
  模型字串的雙前綴。
- **`output_schema` + Pydantic** 才是可靠的結構化輸出；
  在 instruction 裡拜託模型回 JSON 不可靠。

## 動手練習

1. 幫 `Sentiment` 加一個 `needs_followup: bool` 欄位，
   重跑第 4 節，看模型有沒有正確判斷。
2. 把 `temperature` 設成 `0.0` 再跑一次第 4 節的分析，
   連跑三次，確認結構化輸出的穩定度。
3. 拿掉 `output_schema`，改成在 instruction 裡寫「請回傳 JSON」，
   跑五次看看有幾次輸出是可以直接 `json.loads()` 的。

---
**下一站 → `04_runtime_session.ipynb`**：Layer 2 開始——
Runner、Session、State 是怎麼把單次呼叫變成有記憶的對話。